# A Sample Linear Regression Model on Gold Economy

Gold is the resource that turns performance into power in League of Legends. But which in-game actions
actually *earn* that gold? Is it kills? Farming (CS)? Just surviving longer matches? 

This notebook builds a regression model to answer that with easy-to-understand numbers instead of intuition: *how much of a player's end-of-match gold can be explained by in game economy and at what rate?*

## Content

1. Explore how each stat relates to gold before assuming a straight-line relationship
2. Engineer clean, comparable features from raw match data
3. Fit and diagnose an ordinary least squares model: checking *why* it works, not just *that* it works
4. Summary: what predicts gold, and what that implies about current game system and its quantifiable impact in-game.

**A note on data quality:** this analysis excludes matches where the last recorded stat
snapshot was stale (`unlogged_duration != 0`) or where the game ended abnormally early
(`game_duration <= 300` seconds), since both would distort the true relationship between
performance and gold.

In [0]:
%pip install statsmodels

In [0]:
from typing import Callable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import (
    train_test_split, 
    KFold, 
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    StandardScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline

In [0]:
%sql
USE CATALOG league_records;

USE SCHEMA gold;

In [0]:
sns.set_theme(style="dark")

## Table Schema
The following table schema filter for matches where `unlogged_duration` is equals to 0 to prevent stale, unreliable records and also excluding remakes (only `game_duration` > 300).

- `game_duration` INT (match length in seconds)
- `is_blue` INT 1 = 'BLUE' / 0 = 'RED'. Binarize as a bernoulli feature.
- `win` INT 1 = true / 0 = false. Binarize as a bernoulli feature.
- `champion_role` STRING (resolved role: top/jungle/mid/adc/support)
- `level` INT (end-of-match champion level)
- `kills` INT (number of kills by player)
- `deaths` INT (number of deaths by player)
- `assists` INT (number of assists by player)
- `cs` INT (combined lane + jungle creep score)
- `total_gold` INT (cumulative gold earned)

In [0]:
SOURCE = spark.sql(
"""
SELECT *
FROM matchend_player_stats
;
"""
).toPandas()

In [0]:
display(SOURCE.head())

In [0]:
def feature_eng_pipeline(src: pd.DataFrame) -> pd.DataFrame:
    return (src
        .query('unlogged_duration == 0')
        .query('game_duration > 300')
        .assign(win=lambda df: df['win'].astype('int64'))
        [[
            'game_duration', 'team', 'win', 'champion_role',
            'level', 'kills', 'deaths', 'assists', 'cs',
            'total_gold'
        ]]
    )

## Features to Gold Relationship Summary:
1. `cs`: has a support clustering group (< 80cs), strong linear.
2. `kills`: moderately linear.
3. `deaths`: weak linear.
4. `assists`: moderately linear with a support clustering (> 15, low gold)
5. `levels`: strong linear.
6. `game_duration`: strong linear.

In [0]:
sns.pairplot(
    feature_eng_pipeline(SOURCE).sample(100),
    vars=['kills', 'assists', 'cs', 'level', 'game_duration', 'total_gold', 'deaths'],
    hue='win',
    plot_kws={'alpha': 0.5},
)

In [0]:
def custom_ss(src: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    mtx = src[cols].to_numpy()
    x_ = np.mean(mtx, axis=0)
    std_ = np.std(mtx, axis=0, ddof=1)

    res = src.copy()
    ss_mtx = (mtx - x_)/ std_
    res[[f'{x}_ss' for x in cols]] = ss_mtx

    return res.drop(columns=cols)

In [0]:
custom_ss(
    SOURCE.sample(100), 
    ['kills', 'assists', 'cs', 'level', 'game_duration', 'total_gold']
).boxplot(
    column=['kills_ss', 'assists_ss', 'cs_ss', 'level_ss', 'game_duration_ss', 'total_gold_ss'],
    vert=False
)

## Linear Regression
* Response: `total_gold`
* Features: `win`, `role`, `game_duration`, `cs`, `level`, `kills`, `deaths`, `assists`

Preprocessing: 

In [0]:
def sm_linear_reg(X: pd.DataFrame, y: pd.Series, pp: ColumnTransformer = None) -> None:
    if pp is not None:
        X_t = pp.fit_transform(X)
        X = pd.DataFrame(X_t, columns=pp.get_feature_names_out(), index=X.index)

    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type='HC3')
    print(model.summary())

    diag_df = pd.DataFrame({
        'features': model.params.index,
        'coef': model.params.values,
        'pvalue': model.pvalues.values,
    })
    nonsig_features = diag_df.loc[diag_df['pvalue'] > 0.05, 'features']
    print(f'Insignificant features: {nonsig_features}')

    yi = model.predict(X)
    res = (y - yi).to_numpy()

    rng = np.random.default_rng(42)
    mean_, std_ = res.mean(), res.std(ddof=1)
    theoretical_normal = rng.normal(loc=mean_, scale=std_, size=len(res))

    fig, axes = plt.subplots(1, 3, figsize=(15,5))

    sns.histplot(res, stat='probability', ax=axes[0], label='sample residuals', color='#5DCAA5')
    sns.histplot(theoretical_normal, stat='probability', ax=axes[0], alpha=0.35,
                 label='theoretical normal', color='#D85A30')

    for n_std in [2, 4]:
        for sign in [-1, 1]:
            x = mean_ + sign * n_std * std_
            axes[0].axvline(x, color='#534AB7', linestyle='--', linewidth=1, alpha=0.8)
            axes[0].text(x, axes[0].get_ylim()[1] * 0.95, f'{sign*n_std:+d}σ', rotation=90,
                          va='top', ha='right', fontsize=8, color='#534AB7')

    axes[0].set_title('Histogram distribution of residual')
    axes[0].legend()

    sm.qqplot(res, ax=axes[1], line='45', fit=True)
    axes[1].set_title('QQ Plot of residual')

    sns.scatterplot(x=yi, y=res, ax=axes[2])
    axes[2].set_title('Residual over predicted y')

    plt.tight_layout()

In [0]:
def _pp_lookup(pp_features: dict) -> ColumnTransformer:
    pp_lookup = {
        'ss': StandardScaler(),
        'mms': MinMaxScaler(),
        'dummy': OneHotEncoder(drop='first', dtype='int64', sparse_output=False),
    }
    if pp_features is None:
        return ColumnTransformer([], remainder='passthrough')

    if not set(pp_features.keys()).issubset(set(pp_lookup.keys())):
        raise ValueError(
            f'Invalid scaling name, got {[*pp_features.keys()]}. '
            f'Expecting one of {[*pp_lookup.keys()]}.'
        )

    return ColumnTransformer([
        (name, pp_lookup[name], cols)
        for name, cols in pp_features.items()
    ], remainder='passthrough')

In [0]:
def linear_ols(
    # -- Data preparations
    src: pd.DataFrame,
    response: str,
    features: list[str] = None,
    pp_features: dict[str, list[str]] = None,
    # -- Hyperparameters
    estimator: str = 'OLS',
    alpha_grid: list[float] = None,
    # -- Debugging
    seed: int = 42,
    diag_with_sm: bool = False,
):
    # -- 00. Apply feature engineering (previously defined but never called)
    data = feature_eng_pipeline(src)

    # -- 01. Data preparations
    y = data[response]
    X = data[features] if features is not None else data.drop(columns=response)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        shuffle=True,
        random_state=seed,
    )
    steps = []

    # -- 02. Parse and return preprocessor object
    pp = _pp_lookup(pp_features)
    steps.append(('pp', pp))

    # -- 03. Fit statsmodels and check preliminary results (pass pp so strings get encoded)
    sm_linear_reg(X_train, y_train, pp=pp) if diag_with_sm else None

    # -- 04. Build estimator
    match estimator:
        case 'OLS': est = LinearRegression()
        case 'Ridge': est = Ridge()
        case 'Lasso': est = Lasso()
        case _: raise ValueError(f'Invalid model, got {estimator}. Expecting one of ["OLS", "Ridge", "Lasso"]')

    print(f'Using {estimator}...')
    steps.append(('est', est))

    # -- 05. Fit pipeline, hypertuning alpha only when the estimator supports it
    pipeline = Pipeline(steps)
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)

    if estimator == 'OLS':
        model = pipeline
        model.fit(X_train, y_train)
        cv_score = np.mean(cross_val_score(model, X_train, y_train, cv=kf))
    else:
        param_grid = {'est__alpha': alpha_grid or [0, 0.01, 0.1, 0.2, 0.5, 1, 2, 5, 10]}
        model = GridSearchCV(pipeline, param_grid, cv=kf, scoring='r2', refit=True)
        model.fit(X_train, y_train)
        
        cv_score = model.best_score_
        print(f'Best alpha: {model.best_params_["est__alpha"]}')

    # -- 06. Report results on held-out set
    y_pred = model.predict(X_test)
    sst = np.sum((y_test - np.mean(y_test)) ** 2)
    sse = np.sum((y_test - y_pred) ** 2)
    r2 = (sst - sse) / sst
    mse = sse / len(y_test)
    rmse = np.sqrt(mse)

    return {
        'cv_score': cv_score,
        'train_r2': model.score(X_train, y_train),
        'test_r2': r2,
        'mse': mse,
        'rmse': rmse,
    }

In [0]:
linear_ols(
    # -- Data preparations
    src=SOURCE,
    response='total_gold',
    # -- Preprocessor
    pp_features={
        'ss': ['game_duration', 'cs', 'level'],
        'mms': ['kills', 'deaths', 'assists'],
        'dummy': ['champion_role', 'team'],
    },
    # -- Regularization selection
    estimator='Ridge',
    # -- Others
    seed=42,
    diag_with_sm=True,
)

# Gold Economy OLS Summary

The above regression model explains **95.6% of the variation** in a player's end-of-match gold, and holds up on unseen matches (test R² = 0.955, avg. error ≈ 806 gold). Below is what actually moves the needle, translated into real game units.

## Gold impact per unit

| Driver | Gold Impact |
|---|---|
| +1 creep score (CS) | +26 gold |
| +1 minute of game time | +146 gold |
| +1 champion level | +139 gold |
| +1 kill | +368 gold |
| +1 assist | +70 gold |
| +1 death | +40 gold* |
| Winning the match | +274 gold |
| Playing Support (vs. ADC) | +1,200 gold |
| Playing Jungle (vs. ADC) | −599 gold |
| Playing Top (vs. ADC) | −1,297 gold |
| Playing Mid (vs. ADC) | −1,454 gold |
| Blue side vs. Red side | No meaningful difference |

*All effects are statistically significant at alpha = 0.05, except side (Blue/Red)*

## Takeaways

- **Farming (CS) and time played are still the backbone of income**: mechanical, steady gold generation.
- **Kills are the single biggest per-event payout** (368 gold), nearly 5x an assist.
- **Support earns more than its raw stats would suggest**: the model confirms the game's built-in support gold-generation design.
- **Deaths show a small positive association with gold**: not because dying pays out, but because death counts likely track fight involvement (players who fight more, die more *and* earn more kill/assist gold)! 
- **Winning adds a modest independent bump** (+274 gold) beyond what's already explained by performance stats.

Residuals show mild skew, meaning the model slightly underperforms on a subset of matches (likely unusually short games or blowouts).